In [1]:
import json

data = json.load(open("data/pile-ner/train.json"))




In [4]:
data.keys()

dict_keys(['documents', 'metadata'])

In [7]:
data['documents'][0].keys()

dict_keys(['id', 'entities', 'sentences'])

In [17]:
data['documents'][1]['id']

'ner_1'

In [18]:
data['documents'][1]['entities']

[{'type': 'person', 'sentence_idx': 0, 'start_word_idx': 0, 'end_word_idx': 2},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 56,
  'end_word_idx': 58},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 70,
  'end_word_idx': 72},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 94,
  'end_word_idx': 96},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 119,
  'end_word_idx': 121},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 146,
  'end_word_idx': 148},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 185,
  'end_word_idx': 188},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 208,
  'end_word_idx': 210},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 222,
  'end_word_idx': 224},
 {'type': 'person',
  'sentence_idx': 0,
  'start_word_idx': 240,
  'end_word_idx': 242},
 {'type': 'organization',
  'sentence_idx': 0,
  'start_word_idx': 26,
  'end_word_idx': 28},
 {'type': 'organizatio

In [21]:
data['documents'][0]['sentences'][0][9:10]


['lua']

In [15]:
temp = " ".join(i for i in data['documents'][0]['sentences'][0])
print (temp)

Q : Position character based on enemy coordinates in lua I have written a function here which should turn my character based on enemy coordinates but it ' s not perfect because it does not always turn where I want it to and perhaps there is a better way of writing it local myPosition = { x = 350 , y = 355 } local enemyPosition = { x = 352 , y = 354 } local xValue , yValue , xDir , yDir , dir if myPosition . x > enemyPosition . x then xValue = myPosition . x - enemyPosition . x elseif myPosition . x < enemyPosition . x then xValue = myPosition . x - enemyPosition . x else xValue = 0 end if myPosition . y > enemyPosition . y then yValue = myPosition . y - enemyPosition . y elseif myPosition . y < enemyPosition . y then yValue = myPosition . y - enemyPosition . y else yValue = 0 end if xValue < 0 then xDir = " TURN RIGHT " elseif xValue > 0 then xDir = " TURN LEFT " end if yValue < 0 then yDir = " TURN DOWN " elseif yValue > 0 then yDir = " TURN UP " end if xValue > yValue then dir = xDir

In [1]:
"""Main entrypoint for OWNER
"""
from argparse import ArgumentParser
import tempfile
import logging
import os
import tomllib
import mlflow
import torch
from owner.utils.mlflow import log_config, absolutify
from owner.training.mention_detection import MentionDetectionTrainer
from owner.training.entity_typing import EntityTypingTrainer
from owner.training.base import BaseTrainer
from owner.training.ner import NerTrainer


os.environ["TOKENIZERS_PARALLELISM"] = 'false'

logger = logging.getLogger(__name__)

/home/smallp/miniconda3/envs/owner/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config_file = "configs/conll2003-ai.toml"

with open(config_file, 'rb') as file:
    toml_config = tomllib.load(file)
    absolutify(toml_config)
    log_config(toml_config)


model_name = toml_config['model']
save_state = toml_config.get('save_state', 'none')
save_path = toml_config.get('save_path', None)
model: BaseTrainer = None

# Model
match model_name:
    case 'mention_detection':
        model = MentionDetectionTrainer(toml_config)
    case 'entity_typing':
        model = EntityTypingTrainer(toml_config)
    case 'ner':
        model = NerTrainer(toml_config)
    case _:
        raise ValueError(f'Unknown model: "{model_name}"')

# Datasets
# logger.info("Loading and preprocessing data")
if save_state == 'load_finetuned':
    print("Loading data from \"%s\"" % save_path)
    model.load_data(False)
else:
    model.load_data(True)

torch.manual_seed(toml_config['seed'])



/home/smallp/miniconda3/envs/owner/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading data from "/home/smallp/code/OWNER/checkpoints/ner/conll2003/100"


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/home/smallp/miniconda3/envs/owner/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
100%|██████████| 431/431 [00:00<00:00, 1448.41it/s]


In [3]:
# Training and evaluation
if save_state == 'load_finetuned':
    # logger.info(">>> Loading model from \"%s\" <<<", save_path)
    print(f">>> Loading model from \"{save_path}\"")
    model.load_model(save_path)
else:
    # logger.info(">>> Training model <<<")
    model.train()


>>> Loading model from "/home/smallp/code/OWNER/checkpoints/ner/conll2003/100"


In [ ]:


# Training and evaluation
if save_state == 'load_finetuned':
    # logger.info(">>> Loading model from \"%s\" <<<", save_path)
    model.load_model(save_path)
else:
    # logger.info(">>> Training model <<<")
    model.train()

# logger.info(">>> Evaluating model <<<")
model.evaluate()

if save_state == 'save_finetuned':
    # logger.info(">>> Saving model to \"%s\" <<<", save_path)
    model.save_model(save_path)

In [ ]:
def main(config: dict):
    """Main entrypoint
    Args:
        model (str): model to run
        config (dict): args for run
    """
    model_name = config['model']
    save_state = config.get('save_state', 'none')
    save_path = config.get('save_path', None)
    model: BaseTrainer = None

    # Model
    logger.info("Loading model: %s", model_name)
    match model_name:
        case 'mention_detection':
            model = MentionDetectionTrainer(config)
        case 'entity_typing':
            model = EntityTypingTrainer(config)
        case 'ner':
            model = NerTrainer(config)
        case _:
            raise ValueError(f'Unknown model: "{model_name}"')

    # Datasets
    logger.info("Loading and preprocessing data")
    if save_state == 'load_finetuned':
        model.load_data(False)
    else:
        model.load_data(True)

    torch.manual_seed(config['seed'])

    # Training and evaluation
    if save_state == 'load_finetuned':
        logger.info(">>> Loading model from \"%s\" <<<", save_path)
        model.load_model(save_path)
    else:
        logger.info(">>> Training model <<<")
        model.train()

    logger.info(">>> Evaluating model <<<")
    model.evaluate()

    if save_state == 'save_finetuned':
        logger.info(">>> Saving model to \"%s\" <<<", save_path)
        model.save_model(save_path)

### Test data

In [1]:
"""Common model classes
"""
from typing import List, Set
from pydantic import BaseModel, Field

# A sentence is a list of words
Sentence = List[str]


class Entity(BaseModel):
    """Entity
    """
    type: str | int         # Type of the entity
    sentence_idx: int       # Index of the sentence in the document
    # [start_word_idx, end_word_idx[
    start_word_idx: int
    end_word_idx: int


class MiniDocument(BaseModel):
    """Minimal document (for evaluation purpose)
    """
    id: str  # Id of the document (may differ from the index of the document in the dataset)
    entities: List[Entity] = Field(default=[])


class Document(MiniDocument):
    """Document
    """
    sentences: List[Sentence]


##### Dataset #####
class Metadata(BaseModel):
    """Metadata
    """
    entity_types: Set[str | int] = Field(default=set())


class Dataset(BaseModel):
    """Dataset
    """
    documents: List[Document]
    metadata: Metadata


class MiniDataset(BaseModel):
    """Dataset (for evaluation purpose)
    """
    documents: List[MiniDocument]
    metadata: Metadata


In [4]:
from pydantic import BaseModel
from typing import List
class Dataset(BaseModel):
    """Dataset
    """
    documents: List[Document]
    metadata: Metadata


input_file = "data/crossner/ai/ai_test_converted.json"

with open(input_file, 'r', encoding='utf-8') as file:
    
    temp_data = Dataset.model_validate_json(file.read())


In [4]:
help(Dataset.model_validate_json)

Help on method model_validate_json in module pydantic.main:

model_validate_json(json_data: 'str | bytes | bytearray', *, strict: 'bool | None' = None, context: 'dict[str, Any] | None' = None) -> 'Model' class method of __main__.Dataset
    Usage docs: https://docs.pydantic.dev/2.5/concepts/json/#json-parsing
    
    Validate the given JSON data against the Pydantic model.
    
    Args:
        json_data: The JSON data to validate.
        strict: Whether to enforce types strictly.
        context: Extra variables to pass to the validator.
    
    Returns:
        The validated Pydantic model.
    
    Raises:
        ValueError: If `json_data` is not a JSON string.



In [11]:
print (temp_data)

temp_data.documents[0]

documents=[Document(id='0', entities=[Entity(type='algorithm', sentence_idx=0, start_word_idx=5, end_word_idx=8), Entity(type='algorithm', sentence_idx=0, start_word_idx=10, end_word_idx=13), Entity(type='algorithm', sentence_idx=0, start_word_idx=15, end_word_idx=17)], sentences=[['Typical', 'generative', 'model', 'approaches', 'include', 'naive', 'Bayes', 'classifier', 's', ',', 'Gaussian', 'mixture', 'model', 's', ',', 'variational', 'autoencoders', 'and', 'others', '.']]), Document(id='1', entities=[Entity(type='conference', sentence_idx=0, start_word_idx=6, end_word_idx=7), Entity(type='conference', sentence_idx=0, start_word_idx=11, end_word_idx=12), Entity(type='conference', sentence_idx=0, start_word_idx=14, end_word_idx=20)], sentences=[['Finally', ',', 'every', 'other', 'year', ',', 'ELRA', 'organizes', 'a', 'major', 'conference', 'LREC', ',', 'the', 'International', 'Language', 'Resources', 'and', 'Evaluation', 'Conference', '.']]), Document(id='2', entities=[Entity(type='al

Document(id='0', entities=[Entity(type='algorithm', sentence_idx=0, start_word_idx=5, end_word_idx=8), Entity(type='algorithm', sentence_idx=0, start_word_idx=10, end_word_idx=13), Entity(type='algorithm', sentence_idx=0, start_word_idx=15, end_word_idx=17)], sentences=[['Typical', 'generative', 'model', 'approaches', 'include', 'naive', 'Bayes', 'classifier', 's', ',', 'Gaussian', 'mixture', 'model', 's', ',', 'variational', 'autoencoders', 'and', 'others', '.']])

In [18]:
print (temp_data.documents[0])

id='0' entities=[Entity(type='algorithm', sentence_idx=0, start_word_idx=5, end_word_idx=8), Entity(type='algorithm', sentence_idx=0, start_word_idx=10, end_word_idx=13), Entity(type='algorithm', sentence_idx=0, start_word_idx=15, end_word_idx=17)] sentences=[['Typical', 'generative', 'model', 'approaches', 'include', 'naive', 'Bayes', 'classifier', 's', ',', 'Gaussian', 'mixture', 'model', 's', ',', 'variational', 'autoencoders', 'and', 'others', '.']]


In [19]:
temp_data.metadata 

Metadata(entity_types={'product', 'misc', 'country', 'conference', 'task', 'field', 'location', 'programlang', 'metrics', 'algorithm', 'university', 'organisation', 'researcher', 'person'})

In [21]:
len(temp_data.documents)

431

In [ ]:
from owner.data.serialization import from_owner
from owner.evaluation.base import merge_dataset_with_predictions
pred_entities_v2 = from_owner(
        "data/crossner/ai/ai_test_converted.json",
        "cross-ai"
    ).dataset
pred_entities_dataset = merge_dataset_with_predictions(
        test_dataset, pred_entities_v2.documents)

In [12]:
pred_entities_v2

In [17]:
from argparse import ArgumentParser
import tempfile
import logging
import os
import tomllib
from owner.utils.mlflow import log_config, absolutify


config_file = "configs/conll2003-ai.toml"
with open(config_file, 'rb') as file:
        toml_config = tomllib.load(file)
        print (toml_config)
        absolutify(toml_config)
        log_config(toml_config)



{'seed': 100, 'model': 'ner', 'save_state': 'load_finetuned', 'save_path': 'checkpoints/owner/conll2003/100', 'data': {'train_dataset_path': 'data/conll2003/train.json', 'train_dataset_name': 'conll2003', 'test_dataset_path': 'data/crossner/ai/test.json', 'test_dataset_v2_path': 'data/crossner/ai/ai_test_converted.json', 'test_dataset_name': 'crossner-ai'}, 'mention_detection': {'plm_name': 'microsoft/deberta-v3-base', 'max_len': 256, 'batch_size': 32, 'num_epochs': 4, 'learning_rate': 2e-05}, 'entity_typing': {'template': '{sentence} {entity} is a [MASK].', 'plm_name': 'bert-base-uncased', 'max_len': 256, 'batch_size': 128, 'num_epochs': 4, 'learning_rate': 2e-05, 'k_min': 2, 'k_max': 30, 'k_step': 2}}


In [18]:
data_config = toml_config['data']
data_config


{'train_dataset_path': '/home/smallp/code/OWNER/data/conll2003/train.json',
 'train_dataset_name': 'conll2003',
 'test_dataset_path': '/home/smallp/code/OWNER/data/crossner/ai/test.json',
 'test_dataset_v2_path': '/home/smallp/code/OWNER/data/crossner/ai/ai_test_converted.json',
 'test_dataset_name': 'crossner-ai'}